# Variants from Vario output

The script contains steps where data are prepared but need to be prosessed by the shell scripts (vireo.s and vep.sh) to run. 

https://github.com/single-cell-genetics/cellsnp-lite/issues/109

In [1]:
options(warn=-1)

In [2]:
library_load <- suppressMessages(
    
    list(
        
        # Seurat 
        library(Seurat), 
        
        #  Rsamtools 
        library(Rsamtools), 
        
        # Data 
        library(GenomicRanges), 
        library(tidyverse), 
        library(data.table), 
        library(Matrix), 
        
        # Plotting 
        library(ggplot2), 
        library(patchwork), 
        library(ComplexHeatmap), 
        library(viridis), 
        
        # Pyhton compatibility
        library(reticulate)
        
    )
    
)

In [3]:
# Configure reticulate 
use_condaenv(condaenv='p.3.10.16-FD20250213HSCT', conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

python:         /nobackup/peer/fdeckert/miniconda3/envs/p.3.10.16-FD20250213HSCT/bin/python
libpython:      /nobackup/peer/fdeckert/miniconda3/envs/p.3.10.16-FD20250213HSCT/lib/libpython3.10.so
pythonhome:     /nobackup/peer/fdeckert/miniconda3/envs/p.3.10.16-FD20250213HSCT:/nobackup/peer/fdeckert/miniconda3/envs/p.3.10.16-FD20250213HSCT
version:        3.10.16 | packaged by conda-forge | (main, Dec  5 2024, 14:16:10) [GCC 13.3.0]
numpy:          /nobackup/peer/fdeckert/miniconda3/envs/p.3.10.16-FD20250213HSCT/lib/python3.10/site-packages/numpy
numpy_version:  2.1.3

NOTE: Python version was forced by use_python() function

In [4]:
random_seed <- 42
set.seed(random_seed)

In [5]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [6]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

# Load Seurat object for meta data access 

In [7]:
# Store data as h5ad 
sc <- import("scanpy")

adata <- sc$read_h5ad("data/object/qc.h5ad")

cnt <- t(adata$X)
colnames(cnt) <- adata$obs_names$to_list()
rownames(cnt) <- adata$var_names$to_list()

so <- Seurat::CreateSeuratObject(counts=cnt, meta.data=adata$obs)

# Load genotype

In [69]:
# Genotype
genotype_file <- "result/genotype/genotype.csv"
genotype <- read.csv(genotype_file, row.names=1) %>% tibble::rownames_to_column("cell_id") %>% dplyr::left_join(., so@meta.data %>% dplyr::select(cell_id, time_point), by=join_by(cell_id)) %>% na.omit() %>% 
    dplyr::filter(vireo_donor_id %in% c("donor0", "donor1")) %>% dplyr::filter(time_point %in% c("D14", "D100", "GVHD")) %>% dplyr::select(vireo_patient_id, vireo_donor_id, genotype_class) %>% distinct()
genotype <- reshape2::dcast(genotype, vireo_patient_id~vireo_donor_id, value.var="genotype_class") %>% dplyr::rename(PATIENT_ID=vireo_patient_id, GTC_DONOR0=donor0, GTC_DONOR1=donor1)

# Load reference annotation 

In [70]:
gtf <- rtracklayer::import("../reference/genome/GRCh38/Homo_sapiens.GRCh38.110.gtf")

# GT variants import

In [71]:
gt_import <- function(patient_id) {
    
    # Import GT donor file
    gt_vcf <- read.table(paste0("data/genotype/3_vireo/", patient_id, "_vireo/GT_donors.vireo.vcf.gz"), header=FALSE)
    colnames(gt_vcf) <- c("CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO", "FORMAT", "DONOR0", "DONOR1")
    gt_vcf$ID <- paste0(gt_vcf$CHROM, "_", gt_vcf$POS, "_", gt_vcf$REF, "_", gt_vcf$ALT)
    
    # Import variant tsv
    variant_vcf_file <- paste0("data/genotype/5_snp/", patient_id, "/variants.tsv")
    variant_vcf <- read.table(variant_vcf_file, header=TRUE)
    colnames(variant_vcf) <- c("ID", "DONOR0", "DONOR1")
    
    # Filter GT donor by variants 
    gt_vcf$VARIANT <- gt_vcf$ID %in% variant_vcf$ID
    
    # Set variables 
    gt_vcf$GT_DONOR0 <- sapply(strsplit(gt_vcf$DONOR0, ":"), "[", 1) 
    gt_vcf$GT_DONOR1 <- sapply(strsplit(gt_vcf$DONOR1, ":"), "[", 1)
    gt_vcf$AD_DONOR0 <- as.numeric(sapply(strsplit(gt_vcf$DONOR0, ":"), "[", 2))
    gt_vcf$AD_DONOR1 <- as.numeric(sapply(strsplit(gt_vcf$DONOR1, ":"), "[", 2))
    gt_vcf$DP_DONOR0 <- as.numeric(sapply(strsplit(gt_vcf$DONOR0, ":"), "[", 3))
    gt_vcf$DP_DONOR1 <- as.numeric(sapply(strsplit(gt_vcf$DONOR1, ":"), "[", 3))
    
    gt_vcf$AD <- sapply(strsplit(sapply(strsplit(gt_vcf$INFO, ";"), "[", 1), "="), "[", 2) %>% as.numeric() 
    gt_vcf$DP <- sapply(strsplit(sapply(strsplit(gt_vcf$INFO, ";"), "[", 2), "="), "[", 2) %>% as.numeric() 
    gt_vcf$OTH <- sapply(strsplit(sapply(strsplit(gt_vcf$INFO, ";"), "[", 3), "="), "[", 2) %>% as.numeric()
    gt_vcf$CNT <- sum(gt_vcf$AD) + sum(gt_vcf$DP) + sum(gt_vcf$OTH)

    gt_vcf$PATIENT_ID <- patient_id
    
    return(gt_vcf)
    
}

In [72]:
gt_vcf <- lapply(paste0("Patient_", 1:12), gt_import)
gt_vcf <- do.call("rbind", gt_vcf)

In [73]:
# There are some duplicated entries - not sure why yet
gt_vcf <- distinct(gt_vcf)

In [74]:
gt_vcf$PATIENT_ID <- factor(gt_vcf$PATIENT_ID, levels=paste0("Patient_", 1:12))

# Variant effect predictor 

In [79]:
cache <- TRUE

In [80]:
if(!cache) {
    
    for(PATIENT_ID in paste0("Patient_", 1:12)) {
    
        vep_input <- dplyr::filter(gt_vcf, PATIENT_ID==PATIENT_ID, OTH<AD, OTH<DP) %>% 
            dplyr::mutate(START=POS, END=POS, VARIANT=paste0(REF, "/", ALT), STRAND="+") %>% 
            dplyr::select(CHROM, START, END, VARIANT, STRAND) %>% 
            dplyr::arrange(CHROM, START) %>%
            unique() %>% as.data.frame() 

        vep_dir <- paste0("data/genotype/6_vep/", PATIENT_ID)
        vep_file <- paste0(vep_dir, "/gt_variants.tsv")
        if(!dir.exists(vep_dir)) {dir.create(vep_dir)}

        write.table(vep_input, vep_file, quote=FALSE, sep="\t", col.names=FALSE, row.names=FALSE)
    
    }   
    
}

In [81]:
vep_import <- function(PATIENT_ID) {
    
    vep <- read.table(paste0("data/genotype/6_vep/", PATIENT_ID, "/variant_effect_output.txt"))
    colnames(vep) <- toupper(c("id", "Location", "Allele", "Gene", "Feature", "Feature_type", "Consequence", "cDNA_position", "CDS_position", "Protein_position", "Amino_acids", "Codons", "Existing_variation", "Extra"))
    
    # Annotate 
    ref <- as.data.frame(gtf)[, c("gene_id", "gene_name")] %>% na.omit() %>% unique()
    colnames(ref) <- c("GENE", "SYMBOL")
    
    vep <- dplyr::left_join(vep, ref, by=join_by(GENE))
    
    # Fix variant ID
    vep$ID <- gsub("/", "_", vep$ID)
    
    vep <- vep[, c("ID", "ALLELE", "FEATURE_TYPE", "CONSEQUENCE", "SYMBOL")]
    
    # Set patient id
    vep$PATIENT_ID <- PATIENT_ID
    
    return(vep)
}

In [82]:
vep <- lapply(paste0("Patient_", 1:12), vep_import)
vep <- do.call("rbind", vep)

In [83]:
# There are some duplicated entries - not sure why yet
vep <- distinct(vep)

# Combine gt variants and vep 

In [84]:
data <- dplyr::left_join(gt_vcf, vep, by=join_by(ID, PATIENT_ID))

# Vireo manual genotype annotation 

In [85]:
data <- dplyr::left_join(data, genotype, by=join_by(PATIENT_ID))

# Add aligned read count, percentage cells expressing a gene and selected meta data

In [86]:
meta <- dplyr::select(so@meta.data, sample_path, patient_id, gvhd_status) %>% dplyr::distinct()

In [87]:
bam_reads_aligned <- lapply(meta$sample_path, function(sample_path) {
    
    # Get alignment stats
    idxstats <- idxstatsBam(paste0(sample_path, "/possorted_genome_bam.bam"))

    # Sum the number of aligned reads
    total_aligned_reads <- sum(idxstats$mapped)
    
    return(total_aligned_reads)

}
                          )

In [88]:
meta$bam_reads_aligned <- unlist(bam_reads_aligned)

In [89]:
mtx <- GetAssayData(so, assay="RNA", layer="counts")

In [90]:
cpm <- mtx %*% Matrix::Diagonal(x=1 / Matrix::colSums(mtx))
cpm <- 10^6 * cpm

In [91]:
# Add patient id based information 
colnames(meta) <- toupper(colnames(meta))
data <- dplyr::left_join(data, dplyr::group_by(meta, PATIENT_ID, GVHD_STATUS) %>% dplyr::summarise(BAM_READS_ALIGNED=sum(BAM_READS_ALIGNED)) %>% dplyr::ungroup(), by=join_by(PATIENT_ID))

`summarise()` has grouped output by 'PATIENT_ID'. You can override using the
`.groups` argument.


In [92]:
# Add gene level annotation 
data <- dplyr::left_join(data, data.frame(PCT=round(rowSums(mtx>0)/ncol(mtx), 5), CPM=rowMeans(cpm)) %>% tibble::rownames_to_column(., "SYMBOL"), by=join_by(SYMBOL))

# Save final output 

In [93]:
write.csv(data, "result/genotype/gt_vep.csv")